# flashinfer_jit_cache_sm75 smoke test

Built from [FlashInfer's repository](https://github.com/flashinfer-ai/flashinfer) (`flashinfer-jit-cache/` and `flashinfer-jit-cache-provider/`) or installed from FlashInfer's wheel index; not on PyPI.

## Example 1 — the import, the version and what the pack says about itself

In [ ]:
import importlib.metadata
import flashinfer_jit_cache.providers.sm75 as pack

TAG = "sm75"
manifest = pack.get_provider()  # what the index reads through the entry point
print("flashinfer-jit-cache-" + TAG, pack.__version__, "built from", pack.__git_version__[:10])
assert pack.__version__ == importlib.metadata.version("flashinfer-jit-cache-" + TAG) == manifest["version"]
# FlashInfer accepts a pack whose version is its own, optionally followed by a +<CUDA> label; the pack and the flashinfer-jit-cache shim must match exactly.
assert pack.__version__.split("+")[0] == importlib.metadata.version("flashinfer-python").split("+")[0], pack.__version__
assert pack.__version__ == importlib.metadata.version("flashinfer-jit-cache"), pack.__version__
assert manifest["schema_version"] == 1 and manifest["provider_id"] == TAG, manifest["provider_id"]
assert manifest["distribution"] == "flashinfer-jit-cache-" + TAG, manifest["distribution"]
assert manifest["cuda_architectures"] == [TAG], manifest["cuda_architectures"]  # one architecture per pack
assert manifest["modules"] == sorted(set(manifest["modules"])) and "sampling" in manifest["modules"]
print(f"OK -- the pack lists {len(manifest['modules'])} kernel modules for {manifest['cuda_architectures']}")

## Example 2 — every listed kernel module is on disk, and nothing unlisted is

In [ ]:
import os, platform, struct
import flashinfer_jit_cache.providers.sm75 as pack

manifest = pack.get_provider()
folder = manifest["jit_cache_dir"]
assert os.path.isdir(folder), folder
on_disk = sorted(name for name in os.listdir(folder) if os.path.isfile(os.path.join(folder, name, name + ".so")))
assert on_disk == manifest["modules"], (f"listed but missing: {sorted(set(manifest['modules']) - set(on_disk))[:5]}; "
                                        f"on disk but unlisted: {sorted(set(on_disk) - set(manifest['modules']))[:5]}")
expected_machine = {"x86_64": 62, "AMD64": 62, "aarch64": 183, "arm64": 183}[platform.machine()]  # ELF e_machine
total = 0
for name in on_disk:
    path = os.path.join(folder, name, name + ".so")
    with open(path, "rb") as handle:
        header = handle.read(20)
    assert header[:4] == b"\x7fELF", path
    machine = struct.unpack_from("<H", header, 18)[0]
    assert machine == expected_machine, f"{path} is not a {platform.machine()} library (machine {machine})"
    total += os.path.getsize(path)
print(f"OK -- {len(on_disk)} kernel modules on disk, all {platform.machine()} shared libraries, {total / 2**20:.1f} MiB together")

## Example 3 — the pack is registered, and the index and FlashInfer both find it

In [ ]:
import importlib.metadata
import flashinfer_jit_cache
import flashinfer_jit_cache.providers.sm75 as pack
from flashinfer.jit import env as jit_env

TAG = "sm75"
entries = [e for e in importlib.metadata.entry_points(group="flashinfer.jit_cache.providers") if e.name == TAG]
assert [e.value for e in entries] == ["flashinfer_jit_cache.providers." + TAG + ":get_provider"], entries
found = {p.provider_id: p for p in flashinfer_jit_cache.get_jit_cache_providers()}
assert TAG in found, f"the index does not find this pack (it finds {sorted(found)})"
assert found[TAG].modules == frozenset(pack.get_provider()["modules"]) and found[TAG].cuda_architectures == {TAG}
in_flashinfer = {p.provider_id: p for p in jit_env.FLASHINFER_AOT_PROVIDERS}
assert TAG in in_flashinfer, f"FlashInfer rejects this pack (it accepts {sorted(in_flashinfer)})"
artifacts = jit_env.get_aot_artifacts("sampling")  # where FlashInfer would load the sampling kernels from
print("OK -- registered as", entries[0].value, "; found by the index and by FlashInfer;",
      len(artifacts), "pre-built sampling artifact(s) resolve for this machine's GPU")

## Example 4 — a kernel from the pack runs on the GPU with run-time compiling switched off

In [ ]:
import os, subprocess, sys, tempfile, torch
assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
TAG = "sm75"
major, minor = torch.cuda.get_device_capability()
if TAG.rstrip("af") != f"sm{major}{minor}":
    print(f"SKIP: this pack holds kernels for {TAG} only; this GPU is compute capability {major}.{minor}")
else:
    code = (
        "import torch, flashinfer\n"
        "probs = torch.softmax(torch.randn(8, 1000, device='cuda'), dim=-1)\n"
        "picked = flashinfer.sampling.top_k_top_p_sampling_from_probs(probs, 20, 0.9)\n"
        "allowed = probs.topk(20, dim=-1).indices\n"
        "assert all(int(picked[r]) in set(allowed[r].tolist()) for r in range(8))\n"
        "print([l.split()[-1] for l in open('/proc/self/maps') if l.rstrip().endswith('/sampling.so')][0])\n")
    child_env = dict(os.environ, FLASHINFER_DISABLE_JIT="1",  # a missing pre-built kernel raises instead of compiling
                     FLASHINFER_WORKSPACE_BASE=tempfile.mkdtemp(prefix="flashinfer_nb_"))
    child = subprocess.run([sys.executable, "-c", code], env=child_env, capture_output=True, text=True, timeout=600)
    assert child.returncode == 0, child.stderr[-2000:]
    loaded = child.stdout.strip().splitlines()[-1]
    assert f"/flashinfer_jit_cache/providers/{TAG}/jit_cache/sampling/sampling.so" in loaded, loaded
    print("OK -- top-k/top-p sampling ran from the pack's own kernel, nothing compiled:", loaded)